# Time Utilisation: the EU report with the site configuration

The other proof notebook moved the **site** table to the EU configuration. This one goes the other
way: we take the **EU** table and switch each EU rule back to the **site** rule, one at a time.
After every switch we measure how much of the gap between the two reports is gone. The last step is
a table: one row per cause, its effect on every measure.

Data: the tables as Power BI loaded them, from the real `.pbix` files (`downloads/pbix_data/`, made
by `python scripts/helper-pbix-extract.py .`). Set `SITE`, run cell by cell.

## Step 1. Settings

In [ ]:
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format

SITE = 'Hull'          # 'Hull' or 'Nottingham'
DATA = '../downloads/pbix_data/'
END = '2026-09-16'     # the EU file ends on 16 Sep 2026: compare the days before
SITE_GLIDE = {'Hull': '../downloads/Glide Path Targets V7 Add Updates as New Columns.csv',
              'Nottingham': '../downloads/Notts GlidePath Targets Add Updates as New Columns.csv'}

KEYS = ['Date', 'MachineName']
MEASURES = ['OEE Time Total', 'Total Time Total', 'Planned OPT Total']
TARGETS = []
HULL_EU_OUT = ['4 Tonne', '6 Tonne', '6000L', 'Analgesics Processing', 'Analgesics X Press',
               'Bosch Capsulator', 'Dettol 1', 'Evolution', 'Fitten', 'Gaviscon TPR',
               'Gaviscon XP700(A)', 'GG4', 'GG5', 'GG6', 'GG7', 'Lemsip Process',
               'Senokot Press', 'Symex suite', 'Vacumix']
NOTT_TU_KEEP = ['N1', 'N10', 'N11', 'N12', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8', 'N9 Auto',
                'O5', 'O6', 'S04', 'S05', 'S06', 'S08', 'S10', 'S11', 'S12', 'S13', 'T2', 'T7']
NOTT_LOSS_OUT = ['Bosch', 'Fette 23', 'Fette12', 'Fette2', 'Fette20', 'Fette4', 'Fette5',
                 'Gran0', 'S14']

## Step 2. Load the three tables
- **EU visual table** `Time Utilisation EU`: what the EU pages show (all sites, we keep `SITE`).
- **EU site part**: the EU query for this site, just before the union.
- **Site table**: what the site pages show.

In [ ]:
def load(name):
    df = pd.read_parquet(DATA + name + '.parquet')
    return df[df['Date'] < END].copy()

def to_minutes(s):
    if pd.api.types.is_datetime64_any_dtype(s):
        return ((s - pd.Timestamp('1899-12-30')).dt.total_seconds() / 60).round(4)
    return pd.to_numeric(s).astype(float)

eu_union = load('eu__Time_Utilisation_EU')
eu_union = eu_union[eu_union['SiteName'] == SITE].copy()
eu_part = load({'Hull': 'eu__Time_Utilisation_H', 'Nottingham': 'eu__Time_Utilisation_N'}[SITE])
site_tbl = load({'Hull': 'hull__Time_Utilisation', 'Nottingham': 'nott__Time_Utilisation_N'}[SITE])
for df in (eu_union, eu_part, site_tbl):
    for col in MEASURES + TARGETS:
        df[col] = to_minutes(df[col])

def totals(df):
    out = {'rows': len(df)}
    for col in MEASURES + TARGETS:
        out[col] = df[col].sum()
    return pd.Series(out)

start = pd.DataFrame({'EU visual table': totals(eu_union), 'Site table': totals(site_tbl)})
start['gap (EU - site)'] = start['EU visual table'] - start['Site table']
display(start)

## Step 3. Cause 1: the EU union drops the seconds
The EU builds its visual table with a union and reads times as hours:minutes there. We switch back
to the EU site part (still with seconds). Rows must stay the same; minutes change a little.

In [ ]:
effect_seconds = totals(eu_part) - totals(eu_union)
display(effect_seconds.to_frame('effect of switching back (site part - union)'))

## Step 4. Cause 2: the EU machine filter
The site table has rows the EU never loads. They should all be machines on the EU filter list.
We cannot add them to the EU table (they are not in the EU file), so we take them from the site table
and show that they are exactly the missing rows.

In [ ]:
machine = site_tbl['MachineName'].str.strip()
if SITE == 'Hull':
    dropped_by_eu = machine.isin(HULL_EU_OUT)
else:
    dropped_by_eu = ~machine.isin(NOTT_TU_KEEP)

missing = site_tbl[dropped_by_eu]
print(f'Site rows on machines the EU drops: {len(missing):,}')
print(missing['MachineName'].value_counts().to_string())

# Check: after these rows, does any site key remain that the EU does not have?
eu_keys = set(map(tuple, eu_part[KEYS].astype(str).values))
rest = site_tbl[~dropped_by_eu]
rest_only = [k for k in map(tuple, rest[KEYS].astype(str).values) if k not in eu_keys]
print()
print(f'Other site keys missing from the EU: {len(rest_only)}   (expected 0)')
effect_machines = totals(missing)
display(effect_machines.to_frame('effect of adding the dropped machines'))

## Step 5. Cause 3: the targets
This table has no targets.

In [ ]:
eu_site = eu_part.copy()
effect_targets = totals(eu_part) * 0

## Step 6. Cause 4: what is left on the shared rows (mostly the refresh date)
What is left on the rows both reports have. The EU file was refreshed a week earlier, so the
current-month export changed in between. The differences should sit only in the last days.

In [ ]:
a = eu_site.groupby(KEYS, dropna=False)[MEASURES + TARGETS].sum()
b = site_tbl[~dropped_by_eu].groupby(KEYS, dropna=False)[MEASURES + TARGETS].sum()
both = a.join(b, how='inner', lsuffix=' EU', rsuffix=' site')
days = set()
for col in MEASURES + TARGETS:
    diff = both[col + ' EU'].fillna(0) - both[col + ' site'].fillna(0)
    bad = diff[diff.abs() > 1e-3].reset_index()
    if len(bad):
        d = eu_site.merge(bad[KEYS], on=KEYS)['Date'].dt.strftime('%Y-%m-%d')
        days |= set(d)
print('days with a difference:', sorted(days) if days else 'none')
effect_refresh = totals(site_tbl[~dropped_by_eu]) - totals(eu_site) - totals(missing) * 0
effect_refresh['rows'] = len(site_tbl[~dropped_by_eu]) - len(eu_site)
display(effect_refresh.to_frame('effect of the refresh date'))

## Step 7. What makes the difference
Each row is one cause and how much it moves each measure, going from the **EU visual table** to the
**site table**. They add up to the total gap; `not explained` should be 0 or tiny.

In [ ]:
causes = pd.DataFrame({
    '1. EU union step (seconds dropped)': effect_seconds,
    '2. EU machine filter': effect_machines,
    '3. targets: glidepath file and key': effect_targets,
    '4. what is left on shared rows (refresh date)': effect_refresh,
}).T
gap = totals(site_tbl) - totals(eu_union)
causes.loc['sum of causes'] = causes.sum()
causes.loc['total gap (site - EU)'] = gap
causes.loc['not explained'] = gap - causes.loc['sum of causes']
display(causes)

## Step 8. Share of each cause
The same table in percent of the total gap, per measure (empty where the gap is 0).

In [ ]:
share = causes.iloc[:4].div(gap.where(gap.abs() > 1e-9)) * 100
display(share.round(1))